In [1]:
import pandas as pd
import plotly.express as px
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [2]:
file_path = "../england_secondary_schools.parquet" 
df = pd.read_parquet(file_path)

In [3]:
# select all london schools

london_df = df[df["london_sub_region"].isin(["Inner London", "Outer London"])].copy()

In [4]:
# select variables we will focus on

plot_df = london_df.dropna(subset=[
    "percentage_fsm", 
    "ks4_attainment8", 
    "scap_utilisation_pct", 
    "number_of_pupils"
]).copy()

## 1. Bivariate Analysis: Attainment vs. Deprivation

In [5]:
# create an interaction scatter plot

fig = px.scatter(
    plot_df,
    x="percentage_fsm",
    y="ks4_attainment8",
    size="number_of_pupils",          
    color="scap_utilisation_pct",     
    color_continuous_scale="RdBu_r",  
    color_continuous_midpoint=100,    
    hover_name="establishment_name",  
    hover_data={
        "la_name": True,
        "percentage_fsm": ":.1f",
        "ks4_attainment8": ":.1f",
        "scap_utilisation_pct": ":.1f",
        "number_of_pupils": True
    },
    title="London Education Inequality: Attainment vs Disadvantage",
    labels={
        "percentage_fsm": "Free School Meals (FSM) %",
        "ks4_attainment8": "Average Attainment 8 Score",
        "scap_utilisation_pct": "School Utilisation Rate (%)",
        "number_of_pupils": "Total Pupils"
    },
    trendline="ols",                  
    trendline_color_override="black"  
)

fig.update_layout(
    template="simple_white",
    height=700,
    width=1000,
    coloraxis_colorbar=dict(title="Utilisation<br>(>100% Overcrowded)")
)

fig.show()

### Key Insights from the Plot

- The Poverty Penalty (Trendline): The black OLS trendline shows a clear negative correlation between the percentage of students on Free School Meals (FSM) and Attainment 8 scores. This visually confirms that deprivation remains a primary driver of educational inequality in London.

- Demand-Driven Overcrowding (Color): Overcrowded schools (marked in red, >100% capacity) are predominantly found above the trendline. This suggests that overcrowding is a symptom of success: parents gravitate towards high-performing schools regardless of the local socio-economic context.

- Capacity Distribution (Size): The size of the bubbles (total pupils) shows that large secondary schools are common across both wealthy and deprived areas, but the "red" oversubscription is more concentrated in schools that outperform their neighborhood expectations (left upper area).

- Positive Outliers: Schools like St Thomas the Apostle College serve as critical case studies. Despite being in a high-deprivation area (FSM >40%), they achieve scores significantly higher than the regression's prediction (58.0 vs. ~45.0). These schools are also among the most overcrowded (119%), proving that high demand exists even in disadvantaged neighborhoods for quality education.

In [7]:
# fit a glm model

print("\n--- Generalized Linear Model (GLM) Results ---")
formula = (
    "ks4_attainment8 ~ percentage_fsm + scap_utilisation_pct + "
    "C(london_sub_region) + C(type_of_establishment)"
)

model = smf.glm(
    formula=formula, 
    data=plot_df, 
    family=sm.families.Gaussian()
).fit()

print(model.summary())


--- Generalized Linear Model (GLM) Results ---
                 Generalized Linear Model Regression Results                  
Dep. Variable:        ks4_attainment8   No. Observations:                  478
Model:                            GLM   Df Residuals:                      466
Model Family:                Gaussian   Df Model:                           11
Link Function:               Identity   Scale:                          46.814
Method:                          IRLS   Log-Likelihood:                -1591.4
Date:                Tue, 10 Mar 2026   Deviance:                       21816.
Time:                        11:42:41   Pearson chi2:                 2.18e+04
No. Iterations:                     3   Pseudo R-squ. (CS):             0.6042
Covariance Type:            nonrobust                                         
                                                               coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------

### Statistical Confirmation (GLM)

- Significant Predictors:
    - FSM% (p=0.000): Using a Generalized Linear Model (Gaussian family), we found that after controlling for school types and geographical locations, the expected average Attainment 8 score decreases by 0.35 points for every 1% increase in the FSM population.This is the strongest and most consistent predictor in our model.

    - School Type (Academy Sponsor Led, p=0.003): Schools managed by trusts due to historical failure perform 3.8 points lower on average than other types, even when controlling for poverty. This suggests institutional legacy plays a major role in student outcomes.

- Non-Significant Findings:
    - School Utilisation (p=0.170): Statistically, being "overcrowded" has no significant negative impact on grades. This is against the common myth that larger/busier schools inherently provide lower-quality education. It reinforces our visual insight that overcrowding is a result of high performance, not a cause of low performance.

    - Geography (Inner vs. Outer London, p=0.093): There is no absolute "geographic divide" in performance once we control for individual school characteristics and poverty levels.

    - Conclusion & Transition: Our statistical analysis suggests that quality education in London is highly sought after and often "over-subscribed" in deprived areas. This leads to our Spatial Question: Given that these schools shown as "positive outliers" are so crowded, how accessible are they to students in nearby high-deprivation zones via public transport?